![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Brain Company Filing Language Metrics Research

This notebook studies whether SEC filing sentiment helps explain future returns

In [ ]:
qb = QuantBook()
# Anchor the research clock to the start of 2026 for a reproducible history window.
qb.set_start_date(2026, 1, 1)
# Daily bars will have an end_time that matches the following midnight.
qb.settings.daily_precise_end_time = False

### Build a Filing Sentiment Universe

Select US Equities with positive sentiment in both the report and MD&A sections of their latest SEC filing, then inspect the returned universe history.

In [ ]:
def select_assets(data: List[BrainCompanyFilingLanguageMetricsUniverseAll]) -> List[Symbol]:
    # Keep names with positive sentiment in both the report and MD&A sections.
    return [d.symbol for d in data
            if d.report_sentiment and d.report_sentiment.sentiment and d.report_sentiment.sentiment > 0
            and d.management_discussion_analyasis_of_financial_condition_and_results_of_operations
            and d.management_discussion_analyasis_of_financial_condition_and_results_of_operations.sentiment
            and d.management_discussion_analyasis_of_financial_condition_and_results_of_operations.sentiment > 0]

# Add the Brain Company Filing Language Metrics universe.
universe = qb.add_universe(BrainCompanyFilingLanguageMetricsUniverseAll, select_assets)
# Request universe history of the last 90 days.
universe_history = qb.universe_history(universe, qb.time - timedelta(90), qb.time - timedelta(1), flatten=True).rename_axis(['time', 'symbol']).drop(columns=['time'])
# Print the returned shape and columns.
print(f"Shape: {universe_history.shape}")
print(f"Columns: {list(universe_history.columns)}")
universe_history.head()

### Universe Diagnostics

Check how many assets pass the filter each day and summarize the factors.

In [ ]:
# Extract the numeric sentiment score from the nested report-sentiment column.
universe_history['reportsentiment'] = universe_history['reportsentiment'].map(lambda x: x.sentiment if x is not None else None)
# Count selected assets by day.
universe_size = universe_history.reset_index().groupby(['time', 'symbol']).size().groupby('time').size()
print(f"Universe days: {len(universe_size)}")
# Store the selected symbol list.
unique_assets = list(universe_history.index.levels[1].unique())
print(f"Mean universe size per day: {universe_size.mean():.1f}")
print('')
print(universe_history.reportsentiment.describe().map('{:0.3f}'.format))
universe_size.plot(title='Daily Universe Size', ylabel='Universe Size');

### Daily Universe Prices

Fetch daily price history for every symbol that appears in the universe.

In [ ]:
# Extract unique assets
symbols = list(universe_history.index.get_level_values(1).unique())
# Fetch daily historical price metrics using the earliest timestamp available in the index.
history = qb.history(symbols, universe_history.index[0][0] - timedelta(1), qb.time, Resolution.DAILY)
history

### Align Filing Sentiment And Returns

Build a joined table of report sentiment and future returns.

In [ ]:
# Align the factor with the return from the next open to the following open.
dataset = (
    universe_history.reset_index().groupby(['time', 'symbol']).agg(reportsentiment=('reportsentiment', 'mean'))
    .join(history.open.unstack('symbol').sort_index().pct_change(2, fill_method=None).shift(-2).stack().rename('futurereturn'), how='inner')
)

dataset.head()